# VPI Defense — Colab runner

Ordered cells for a Colab session. Run top to bottom. State survives session
expiry because everything writes to Google Drive.

**What this notebook does**

1. Mount Drive and set up the project location
2. Install the package (editable) and run unit tests
3. Run the end-to-end integration check on synthetic data
4. Load your Gemini API key from Colab Secrets
5. **Ingest** real images (DocVQA + DocLayNet)
6. **Generate variants** with 4 injection modifications
7. **Analyze with Gemini 2.5 Flash** and archive raw responses
8. Preview results

You need: Gemini API key (Google AI Studio, free — no card required). No GPU required.

## 1. Mount Drive and set project location

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/vpi_defense'

import os, pathlib
pathlib.Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
%cd $PROJECT_ROOT
!ls -la

## 2. Get the project code into `PROJECT_ROOT`

**Option A: upload the vpi_defense folder to Drive manually.**
Copy the entire `vpi_defense/` directory (with `src/`, `tests/`, `scripts/`,
`pyproject.toml`, etc.) into `/content/drive/MyDrive/vpi_defense`. The
cell above already `cd`'d there.

**Option B: clone from your GitHub repo.**  
Uncomment and edit the URL below:

In [ ]:
# !git clone https://github.com/<your-user>/vpi_defense.git .
!ls -la

## 3. Install the package and run unit tests

Editable install so any code edits you make in Drive take effect immediately.

In [ ]:
!pip install -e '.[dev]' --quiet

**CJK font install** — Colab 이미지에 fonts-noto-cjk가 없는 세션이 있어서, M3(한국어 variant 렌더링) 전에 실행. 이미 있으면 즉시 종료.

In [ ]:
!apt-get install -y fonts-noto-cjk --quiet 2>&1 | tail -3
!fc-cache -fv > /dev/null 2>&1
!ls /usr/share/fonts/opentype/noto/ 2>/dev/null | head -5

In [ ]:
!pytest tests/ -q
# Expected: 47 passed

## 4. Sanity: run the synthetic-data integration check

In [ ]:
!python scripts/verify_pipeline.py
# Expected: 'PIPELINE VERIFIED END-TO-END'

## 5. Load your Gemini API key from Colab Secrets

In the left sidebar, click the 🔑 icon and add a secret named
**`GEMINI_API_KEY`** with your key from https://aistudio.google.com/.
Then run:

In [ ]:
from google.colab import userdata
import os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('key set:', bool(os.environ.get('GEMINI_API_KEY')))

## 6. Ingest — download a small seed from HuggingFace

First run downloads the datasets; subsequent runs use HF's cache.
DocLayNet field names can drift — the second cell inspects one sample so
you can compare against what our adapter expects.

In [ ]:
!mkdir -p data/raw
!python -m vpi_defense.ingest --dataset docvqa    --n 10 --split validation --out data/raw

In [ ]:
# Verify DocLayNet-v1.2 field names BEFORE running the ingester.
# If keys or metadata shape differ, patch src/vpi_defense/ingest/doclaynet.py.
from datasets import load_dataset
ds = load_dataset('ds4sd/DocLayNet-v1.2', split='train', streaming=True)
sample = next(iter(ds))
print('keys:', list(sample.keys()))
meta = sample.get('metadata')
print('metadata type:', type(meta).__name__)
print('metadata:', meta)

In [ ]:
!python -m vpi_defense.ingest --dataset doclaynet --n 10 --split train --out data/raw
!ls data/raw/docvqa | head -5
!ls data/raw/doclaynet | head -5
!wc -l data/raw/index.jsonl

## 7. Generate variants (4 modifications × N images)

In [ ]:
!python -m vpi_defense.variants \
    --in  data/raw/index.jsonl \
    --raw data/raw \
    --out data/variants \
    --n-per-image 4 \
    --seed 42
!wc -l data/variants/variants_index.jsonl data/variants/regions_index.jsonl

In [ ]:
# Preview: show one base image and its four variants side by side.
import json, itertools
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

first_img_id = json.loads(next(open('data/raw/index.jsonl')))['image_id']
print('base image_id:', first_img_id)

variants = [json.loads(l) for l in open('data/variants/variants_index.jsonl')]
variants_of_first = [v for v in variants if v['base_image_id'] == first_img_id]

display(Markdown('### Base image'))
base_path = Path('data/raw') / next(json.loads(l)['image_path']
                                    for l in open('data/raw/index.jsonl')
                                    if json.loads(l)['image_id'] == first_img_id)
display(Image.open(base_path))

for v in variants_of_first:
    display(Markdown(f"### {v['modification_type']}"))
    display(Image.open(Path('data/variants') / v['image_path']))

## 8. Analyze with Gemini 2.5 Flash

First-time only: install `google-genai`. Then run analysis on raw images
and variants separately. Raw responses are archived under
`data/analysis/raw_responses/` so we can replay them without the API later.

**If you previously ran this section with Claude and got 400 errors**, the
next cell (7.5) clears the empty result files so we start fresh.

In [ ]:
# Clear any empty output files from a previous failed Claude run.
# Safe to run even on a fresh notebook.
!rm -f data/analysis/regions_raw.jsonl data/analysis/regions_variants.jsonl
!rm -rf data/analysis/raw_responses
!mkdir -p data/analysis/raw_responses
!echo cleared

In [ ]:
!pip install google-genai --quiet

In [ ]:
!mkdir -p data/analysis/raw_responses
!python -m vpi_defense.analysis \
    --provider gemini \
    --in   data/raw/index.jsonl \
    --root data/raw \
    --out  data/analysis/regions_raw.jsonl \
    --save-raw-dir data/analysis/raw_responses

In [ ]:
!python -m vpi_defense.analysis \
    --provider gemini \
    --in         data/variants/variants_index.jsonl \
    --root       data/variants \
    --input-kind variant \
    --out        data/analysis/regions_variants.jsonl \
    --save-raw-dir data/analysis/raw_responses

## 9. Inspect a Gemini raw response

This is what to share back so we can tune the prompt / parser.
Each file contains the JSON array Gemini emitted for that image.

In [ ]:
import json
from pathlib import Path
for p in list(Path('data/analysis/raw_responses').iterdir())[:2]:
    print('====', p.name, '====')
    raw = p.read_text()
    try:
        parsed = json.loads(raw)
        print(json.dumps(parsed, indent=2, ensure_ascii=False)[:1500])
    except Exception:
        print(raw[:1500])

In [ ]:
# Region type distribution — sanity check the region_type heuristic.
from collections import Counter
import json
for name in ('regions_raw', 'regions_variants'):
    path = f'data/analysis/{name}.jsonl'
    types = Counter(json.loads(l)['region_type'] for l in open(path))
    print(f'{name}: {dict(types)}')

## 10. What to share back

After this notebook completes successfully, share:

1. The output of cell 6.2 (DocLayNet field names) — confirms the ingester
   assumption.
2. One or two files from `data/analysis/raw_responses/` — confirms
   Gemini's response shape and how well it follows the `kind` schema.
3. The region type distribution printed in cell 9.2.

With those, we can tune the prompt/parser and move on to M6 (risk
function + IG).